# ICS40125 - Laboratorio N°03

Manipulación avanzada de datos con pandas sobre el catálogo de Netflix.

In [ ]:
import pandas as pd

df = pd.read_csv('https://raw.githubusercontent.com/fralfaro/ICS40125/main/docs/labs/data/netflix_titles.csv')
df.head()

## Parte 1: Limpieza y preparación

In [ ]:
# 1. Descripcion general del dataset
print("Filas y columnas:", df.shape)
print()
print("Tipos de datos:")
print(df.dtypes)
print()
print("Nulos por columna:")
print(df.isnull().sum())

In [ ]:
# 2. Convertir date_added a tipo fecha
df['date_added'] = pd.to_datetime(df['date_added'].str.strip(), format='%B %d, %Y', errors='coerce')
df['date_added'].head()

In [ ]:
# 3. Columnas auxiliares con assign
df = df.assign(
    year_added=df['date_added'].dt.year,
    month_added=df['date_added'].dt.month
)
df[['title', 'date_added', 'year_added', 'month_added']].head()

## Parte 2: Técnicas avanzadas de pandas

In [ ]:
# 4. Peliculas agregadas despues de 2018
peliculas_recientes = df.loc[(df['type'] == 'Movie') & (df['year_added'] > 2018)]
peliculas_recientes[['title', 'year_added']].head()

In [ ]:
# 5. str.contains y str.extract
# titulos que contienen 'love'
con_love = df.loc[df['title'].str.contains('love', case=False, na=False)]
print("Titulos con 'love':", len(con_love))

# duracion en minutos (solo peliculas)
df['duration_min'] = df['duration'].str.extract(r'(\d+)\s*min').astype(float)
df[['title', 'duration', 'duration_min']].head()

In [ ]:
# 6. explode sobre listed_in (un genero por fila)
df_generos = df.assign(genre=df['listed_in'].str.split(', ')).explode('genre')
df_generos[['title', 'genre']].head()

In [ ]:
# 7. Top 10 generos mas frecuentes
top_generos = df_generos['genre'].value_counts().head(10)
top_generos

In [ ]:
# 8. where / mask: marcar peliculas de mas de 120 min como "largo"
df['contenido_largo'] = df['duration_min'].mask(df['duration_min'] > 120, 'largo').where(df['duration_min'] > 120, 'normal')
df.loc[df['duration_min'].notna(), ['title', 'duration_min', 'contenido_largo']].head()

In [ ]:
# 9. Filtrar: > 100 min, rating 'R', pais 'United States'
filtro = df.loc[
    (df['duration_min'] > 100) &
    (df['rating'] == 'R') &
    (df['country'] == 'United States')
]
filtro[['title', 'duration_min', 'rating', 'country']].head()

In [ ]:
# 10. Top 10 peliculas mas largas con .style
top_largas = (
    df.loc[df['type'] == 'Movie', ['title', 'duration_min']]
    .dropna()
    .sort_values('duration_min', ascending=False)
    .head(10)
)
top_largas.style.background_gradient(subset=['duration_min'], cmap='Reds')

### Pregunta Desafío

In [ ]:
# 11. Combinaciones mas frecuentes de genero y rating
df_gr = df.assign(genre=df['listed_in'].str.split(', ')).explode('genre')
df_gr[['genre', 'rating']].value_counts().head(10)

### Bonus: duplicados y limpieza

In [ ]:
# 12. Peliculas con mismo title pero distinto release_year
repetidos = df.groupby('title')['release_year'].nunique()
repetidos[repetidos > 1]

In [ ]:
# 13. Titulos unicos
print("Titulos unicos:", df['title'].nunique())